In [1]:
import numpy as np

In [125]:
x = np.arange(12).reshape(2, 3, 2)
We = np.ones((4, 2, 2))
Wg = np.ones((2, 4))
top_k = 1

In [126]:
x

array([[[ 0,  1],
        [ 2,  3],
        [ 4,  5]],

       [[ 6,  7],
        [ 8,  9],
        [10, 11]]])

In [127]:
We

array([[[1., 1.],
        [1., 1.]],

       [[1., 1.],
        [1., 1.]],

       [[1., 1.],
        [1., 1.]],

       [[1., 1.],
        [1., 1.]]])

In [128]:
Wg

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [129]:
Wg.shape

(2, 4)

In [130]:
n_batch, l_seq, d_model = x.shape
x_flat = x.reshape(-1, d_model)

In [131]:
n_batch, l_seq, d_model

(2, 3, 2)

In [132]:
x_flat.shape

(6, 2)

In [133]:
x_flat

array([[ 0,  1],
       [ 2,  3],
       [ 4,  5],
       [ 6,  7],
       [ 8,  9],
       [10, 11]])

In [134]:
# Compute gating logits and softmax scores
logits = x_flat @ Wg
logits

array([[ 1.,  1.,  1.,  1.],
       [ 5.,  5.,  5.,  5.],
       [ 9.,  9.,  9.,  9.],
       [13., 13., 13., 13.],
       [17., 17., 17., 17.],
       [21., 21., 21., 21.]])

In [135]:
# Top-k gating: get top-k expert indices and scores
topk_idx = np.argpartition(-logits, top_k - 1, axis=1)[:, :top_k]
topk_logits = np.take_along_axis(logits, topk_idx, axis=1)

In [136]:
topk_idx, topk_logits

(array([[0],
        [0],
        [0],
        [0],
        [0],
        [0]]),
 array([[ 1.],
        [ 5.],
        [ 9.],
        [13.],
        [17.],
        [21.]]))

In [137]:
# Softmax over top-k logits
max_logits = np.max(topk_logits, axis=1, keepdims=True)
exp_logits = np.exp(topk_logits - max_logits)
gate_scores = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)  # shape: (tokens, top_k)
gate_scores

array([[1.],
       [1.],
       [1.],
       [1.],
       [1.],
       [1.]])

In [138]:
# Initialize output
output = np.zeros_like(x_flat, dtype=np.float64)  # shape: (n_batch * l_seq, d_model)
output

array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [139]:
i=0

In [140]:
expert_idx = topk_idx[:, i]  # expert index per token
gate = gate_scores[:, i]     # gate value per token

In [141]:
expert_idx, gate

(array([0, 0, 0, 0, 0, 0]), array([1., 1., 1., 1., 1., 1.]))

In [142]:
n_experts = We.shape[0]
n_experts

4

In [143]:
for expert_id in range(n_experts):
    token_mask = (expert_idx == expert_id)
    print(token_mask)
    if not np.any(token_mask):
        continue
    x_selected = x_flat[token_mask]                  # (num_tokens_i, d_model)
    print(x_selected)
    gate_selected = gate[token_mask][:, None].astype(np.float64)        # (num_tokens_i, 1)
    print(gate_selected)
    print(We[expert_id])
    expert_output = x_selected @ We[expert_id]       # (num_tokens_i, d_model)
    print(expert_output)
    output[token_mask] += gate_selected * expert_output
    print(output)

[ True  True  True  True  True  True]
[[ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]]
[[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]
[[1. 1.]
 [1. 1.]]
[[ 1.  1.]
 [ 5.  5.]
 [ 9.  9.]
 [13. 13.]
 [17. 17.]
 [21. 21.]]
[[ 1.  1.]
 [ 5.  5.]
 [ 9.  9.]
 [13. 13.]
 [17. 17.]
 [21. 21.]]
[False False False False False False]
[False False False False False False]
[False False False False False False]


In [122]:
import numpy as np

def moe(x: np.ndarray, We: np.ndarray, Wg: np.ndarray, n_experts: int, top_k: int) -> np.ndarray:
    """
    Mixture-of-Experts forward pass with Top-K gating (no noise for simplicity).

    Args:
        x: Input tensor of shape (n_batch, l_seq, d_model)
        We: Expert weights of shape (n_experts, d_model, d_model)
        Wg: Gating weights of shape (d_model, n_experts)
        n_experts: Number of experts
        top_k: Number of experts to route each token to

    Returns:
        Output tensor of shape (n_batch, l_seq, d_model)
    """
    n_batch, l_seq, d_model = x.shape
    x_flat = x.reshape(-1, d_model)  # shape: (n_batch * l_seq, d_model)

    # Compute gating logits and softmax scores
    logits = x_flat @ Wg  # shape: (n_batch * l_seq, n_experts)

    # Top-k gating: get top-k expert indices and scores
    topk_idx = np.argpartition(-logits, top_k - 1, axis=1)[:, :top_k]
    topk_logits = np.take_along_axis(logits, topk_idx, axis=1)

    # Softmax over top-k logits
    max_logits = np.max(topk_logits, axis=1, keepdims=True)
    exp_logits = np.exp(topk_logits - max_logits)
    gate_scores = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)  # shape: (tokens, top_k)

    # Initialize output
    output = np.zeros_like(x_flat, dtype=np.float64)  # shape: (n_batch * l_seq, d_model)

    # Route each token to top-k experts
    for i in range(top_k):
        expert_idx = topk_idx[:, i]  # expert index per token
        gate = gate_scores[:, i]     # gate value per token

        # Collect tokens for the current expert
        for expert_id in range(n_experts):
            token_mask = (expert_idx == expert_id)
            if not np.any(token_mask):
                continue
            # Routes each token to its top-k expert
            x_selected = x_flat[token_mask]                  # (num_tokens_i, d_model)
            gate_selected = gate[token_mask][:, None]        # (num_tokens_i, 1)
            expert_output = x_selected @ We[expert_id]       # (num_tokens_i, d_model)

            # combines expert outputs weighted by gating probabilities
            output[token_mask] += gate_selected * expert_output

    return output.reshape(n_batch, l_seq, d_model)


In [124]:
moe(x, We, Wg, n_experts=4, top_k = 2)

array([[[ 1.,  1.],
        [ 5.,  5.],
        [ 9.,  9.]],

       [[13., 13.],
        [17., 17.],
        [21., 21.]]])

**How It Works**
- Gating: Computes the expert scores for each token via Wg, then selects the top_k experts.
- Softmax: Applies softmax only over the top-k scores for routing probabilities.
- Dispatching: Routes each token to its top-k experts, and combines expert outputs weighted by gating probabilities.

In [115]:
x = np.arange(12).reshape(2, 3, 2)
We = np.ones((4, 2, 2))
Wg = np.ones((2, 4))
top_k = 2
n_experts = 4

In [52]:
X_flatten = x.reshape(-1, 2)

In [53]:
X_flatten.shape

(6, 2)

In [54]:
logits = X_flatten @ Wg
logits

array([[ 1.,  1.,  1.,  1.],
       [ 5.,  5.,  5.,  5.],
       [ 9.,  9.,  9.,  9.],
       [13., 13., 13., 13.],
       [17., 17., 17., 17.],
       [21., 21., 21., 21.]])

In [76]:
# top k selection
H_topk = np.full_like(logits, -np.inf)  # Initialize with -inf
print(H_topk)
# Get indices of top-k elements in each row
topk_indices = np.argpartition(-logits, kth=2-1, axis=1)[:, :2]
print(topk_indices)
# Assign top-k values to their positions
for i in range(logits.shape[0]):
    H_topk[i, topk_indices[i]] = logits[i, topk_indices[i]]

[[-inf -inf -inf -inf]
 [-inf -inf -inf -inf]
 [-inf -inf -inf -inf]
 [-inf -inf -inf -inf]
 [-inf -inf -inf -inf]
 [-inf -inf -inf -inf]]
[[0 1]
 [0 1]
 [0 1]
 [0 1]
 [0 1]
 [0 1]]


In [77]:
H_topk

array([[  1.,   1., -inf, -inf],
       [  5.,   5., -inf, -inf],
       [  9.,   9., -inf, -inf],
       [ 13.,  13., -inf, -inf],
       [ 17.,  17., -inf, -inf],
       [ 21.,  21., -inf, -inf]])

In [84]:
H_topk = np.take_along_axis(logits, topk_idx, axis=1)
H_topk

array([[ 1.,  1.],
       [ 5.,  5.],
       [ 9.,  9.],
       [13., 13.],
       [17., 17.],
       [21., 21.]])

In [85]:
exp_logits = np.exp(H_topk - np.max(H_topk, axis = -1, keepdims = True))
print(exp_logits)
gate_score = exp_logits/np.sum(exp_logits, axis=-1, keepdims=True)

[[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]


In [86]:
print(gate_score)

[[0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]]


In [109]:
output = np.zeros_like(x_flat, dtype =np.float64)
output

array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [92]:
expert_idx = topk_idx[:, 0]  # expert index per token
print(expert_idx)
gate = gate_scores[:, 0]
print(gate)

[0 0 0 0 0 0]
[0.5 0.5 0.5 0.5 0.5 0.5]


In [89]:
expert_idx

array([0, 0, 0, 0, 0, 0])

In [101]:
token_mask = (expert_idx == 0)
print(token_mask)
x_selected = x_flat[token_mask] 
print(x_selected)
gate_selected = gate[token_mask][:, None]
print(gate_selected)
expert_output = x_selected @ We[expert_id]
print(expert_output)
output[token_mask] +=  gate_selected * expert_output
print(output)

[ True  True  True  True  True  True]
[[ 0  1]
 [ 2  3]
 [ 4  5]
 [ 6  7]
 [ 8  9]
 [10 11]]
[[0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]
 [0.5]]
[[ 1.  1.]
 [ 5.  5.]
 [ 9.  9.]
 [13. 13.]
 [17. 17.]
 [21. 21.]]
[[ 0.5  0.5]
 [ 2.5  2.5]
 [ 4.5  4.5]
 [ 6.5  6.5]
 [ 8.5  8.5]
 [10.5 10.5]]


In [102]:
gate_selected = gate[token_mask]
gate_selected

array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5])

In [105]:
gate_selected = gate[token_mask][:, None]
gate_selected

array([[0.5],
       [0.5],
       [0.5],
       [0.5],
       [0.5],
       [0.5]])

In [99]:
We[0]

array([[1., 1.],
       [1., 1.]])

In [107]:
top_k

2

In [113]:
for i in range(top_k):
    expert_idx = topk_idx[:, i]  # expert index per token
    print(expert_idx)
    gate = gate_scores[:, i]
    #print(gate)
    for expert_id in range(n_experts):
        token_mask = (expert_idx == expert_id)
        print(token_mask)
        if not np.any(token_mask):
            continue
        #print(token_mask)
        x_selected = x_flat[token_mask] 
        #print(x_selected)
        gate_selected = gate[token_mask][:, None]
        #print(gate_selected)
        expert_output = x_selected @ We[expert_id]
        #print(expert_output)
        output[token_mask] +=  gate_selected * expert_output
        print(output)

[0 0 0 0 0 0]
[ True  True  True  True  True  True]
[[ 0.5  0.5]
 [ 2.5  2.5]
 [ 4.5  4.5]
 [ 6.5  6.5]
 [ 8.5  8.5]
 [10.5 10.5]]
[False False False False False False]
[False False False False False False]
[False False False False False False]
[1 1 1 1 1 1]
[False False False False False False]
[ True  True  True  True  True  True]
[[ 1.  1.]
 [ 5.  5.]
 [ 9.  9.]
 [13. 13.]
 [17. 17.]
 [21. 21.]]
[False False False False False False]
[False False False False False False]


In [ ]:
import numpy as np 
np.random.seed(42) 
d_model = 2 
n_experts = 4 
l_seq = 3 
n_batch = 2 
top_k = 2


In [154]:
x = np.random.rand(n_batch, l_seq, d_model) 
print(x)
We = np.zeros((n_experts, d_model, d_model))
print(We)
Wg = np.random.rand(d_model, n_experts) 
print(Wg)
output = moe(x, We, Wg, n_experts, top_k)

[[[0.74918366 0.94346301]
  [0.27833619 0.95004651]
  [0.38731647 0.58003319]]

 [[0.2262568  0.33912493]
  [0.42993273 0.19611072]
  [0.02596887 0.88682208]]]
[[[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]]]
[[0.9426199  0.53843486 0.27243161 0.91012964]
 [0.62422177 0.83676254 0.08125223 0.96234549]]
[[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]


In [155]:
output

array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])